In [ ]:
import os
from catmap import ReactionModel

# 设置 CatMAP 模型所在目录
model_dir = r'H:\catmap\catmap\lyq_alkene_epoxidation\2-creating_microkinetic_model'

os.chdir(model_dir)

# 检查当前目录
print("当前工作目录：", os.getcwd())
print("MKM 文件存在：", os.path.exists('alkene_epoxidation.mkm'))
print("DFT 能量文件存在：", os.path.exists('energies_dft_neb.txt'))

# 加载原始八步反应模型
model = ReactionModel(setup_file='alkene_epoxidation.mkm')

# Avoid requesting the same map twice.
for variable in ('production_rate', 'rate', 'rate_control',
                 'coverage', 'selectivity_control', 'rxn_order'):
    if variable not in model.output_variables:
        model.output_variables.append(variable)

# Calculate or reload the existing CatMAP solution.
model.run()

# =====================================================
# Restore the original seven figures using native CatMAP plotting.
# All PDFs are saved in 2-creating_microkinetic_model/.
# =====================================================

from catmap import analyze
vm = analyze.VectorMap(model)
vm.plot_variable = 'rate' #tell the model which output to plot
vm.log_scale = True #rates should be plotted on a log-scale
vm.min = 1e-40 # keep very small rates visible in the new DFT run
vm.max = 1e-20 # current DFT model produces very small net rates
vm.plot(save='rate.pdf') #draw the plot and save it as "rate.pdf"

vm.unique_only = False
vm.plot(save='all_rates.pdf')
vm.unique_only = True

vm.production_rate_map = model.production_rate_map #attach map
vm.threshold = 1e-40 # do not hide almost all of the present small rates
vm.plot_variable = 'production_rate'
vm.plot(save='production_rate.pdf')

vm.descriptor_labels = ['OOH formation energy [eV]', 'O formation energy [eV]']
vm.subplots_adjust_kwargs = {'left':0.2,'right':0.8,'bottom':0.15}
vm.plot(save='pretty_production_rate.pdf')

vm.plot_variable = 'coverage'
vm.log_scale = False
vm.min = 0
vm.max = 1
vm.plot(save='coverage.pdf')

vm.include_labels = ['OOH_s', 'O_s']
vm.plot(save='OOH_O_coverage.pdf')

sa = analyze.ScalingAnalysis(model)
sa.plot(save='scaling.pdf')

print('绘图完成：rate.pdf, all_rates.pdf, production_rate.pdf,')
print('pretty_production_rate.pdf, coverage.pdf, OOH_O_coverage.pdf, scaling.pdf')


In [ ]:

# -*- coding: utf-8 -*-

from pathlib import Path

# =====================================================
# 1. 设置需要提取的结果
# =====================================================

# 可修改为：
# 'production_rate'
# 'rate'
# 'rate_control'
# 'coverage'
# 'selectivity_control'
# 'rxn_order'

output_variable = 'production_rate'

# =====================================================
# 2. 直接使用第一段已经运行的 CatMAP 模型
# =====================================================

if 'model' not in globals():
    raise RuntimeError("请先运行第一段代码，完成 CatMAP 模型计算。")

map_name = output_variable + '_map'

if not hasattr(model, map_name):
    raise ValueError(
        f"模型中不存在 {map_name}，"
        f"请检查第一段的 output_variables 并重新运行。"
    )

# =====================================================
# 3. 获取输出变量及标签
# =====================================================

if output_variable == 'rate_control':
    dim = 2
else:
    dim = 1

labels = model.output_labels[output_variable]

def flatten_2d(output):
    """将二维结果展开为一维。"""
    flat = []
    for row in output:
        flat += list(row)
    return flat

# 展开 rate_control 的二维标签
if output_variable == 'rate_control':
    flat_labels = []

    for i in labels[0]:
        for j in labels[1]:
            flat_labels.append('d' + i + '/d' + j)

    labels = flat_labels

# 处理基元反应标签
if output_variable in [
    'rate',
    'rate_constant',
    'forward_rate_constant',
    'reverse_rate_constant'
]:
    str_labels = []

    for label in labels:
        states = ['+'.join(s) for s in label]

        if len(states) == 2:
            new_label = '<->'.join(states)
        else:
            new_label = states[0] + '<->' + states[1] + '->' + states[2]

        str_labels.append(new_label)

    labels = str_labels

# =====================================================
# 4. 生成 TXT 表格
# =====================================================

header = (
    ['descriptor-' + d for d in model.descriptor_names]
    + list(labels)
)

table_lines = ['\t'.join(header)]

result_map = getattr(model, map_name)

for pt, output in result_map:

    if dim == 2:
        output = flatten_2d(output)

    # 描述符与输出结果分别转换，避免数组直接相加
    values = list(pt) + list(output)

    row = [str(float(value)) for value in values]

    if len(row) != len(header):
        raise ValueError(
            f"列数不一致：实际 {len(row)} 列，"
            f"预期 {len(header)} 列。"
        )

    table_lines.append('\t'.join(row))

# =====================================================
# 5. 保存结果
# =====================================================

output_file = Path(output_variable + '_table.txt')

output_file.write_text(
    '\n'.join(table_lines) + '\n',
    encoding='utf-8'
)

print("数据导出完成！")
print("输出变量：", output_variable)
print("结果数量：", len(result_map))
print("保存路径：", output_file.resolve())


In [ ]:

# -*- coding: utf-8 -*-

# =====================================================
# 1. 检查 CatMAP 模型是否已运行
# =====================================================

if 'model' not in globals():
    raise RuntimeError("请先运行第一段代码，完成 CatMAP 模型计算。")

if not hasattr(model, 'coverage_map'):
    raise ValueError(
        "模型中不存在 coverage_map，"
        "请在第一段的 output_variables 中添加 coverage。"
    )

# =====================================================
# 2. 获取各吸附中间体的名称
# =====================================================

labels = model.output_labels['coverage']

print("表面吸附中间体：")
print(labels)

print("=" * 70)

# =====================================================
# 3. 遍历所有描述符点，输出覆盖度
# =====================================================

for descriptors, cvg in model.coverage_map:

    print("描述符坐标：", [float(x) for x in descriptors])

    print("中间体名称：", labels)

    print("表面覆盖度：", [float(c) for c in cvg])

    print("-" * 70)

print("覆盖度数据提取完成！")
